# DeepKey one-timestep LSTM baseline

This notebook compares a compact Keras LSTM with the existing MLP using a stratified 80/20 validation split drawn only from Session 1. Session 2 is never used for model selection or evaluation here. The scaler is fitted on the fit partition only.

Each CSV row contains 46 already-computed timing features. It is reshaped to `(samples, 1, 46)`, so this is a **one-timestep LSTM baseline**, not a model of multiple keystrokes or a continuous keystroke sequence. The recurrent layer therefore cannot learn transitions across successive keystrokes.

The current Windows environment has TensorFlow installed, but its import is blocked by Application Control (`_pywrap_sanitizers` DLL). Do not run this notebook until an authorized working TensorFlow environment is selected. No packages are installed or modified by this notebook.

In [2]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_DIR / 'data' / 'raw' / 'fixed-text.csv'
MLP_RESULTS_PATH = PROJECT_DIR / 'results' / 'metrics' / 'mlp_validation_experiments.csv'
MODEL_OUTPUT_PATH = PROJECT_DIR / 'models' / 'deepkey_lstm_one_timestep.keras'
METRICS_OUTPUT_PATH = PROJECT_DIR / 'results' / 'metrics' / 'lstm_one_timestep_validation.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Raw fixed-text dataset not found: {DATA_PATH}')
if not MLP_RESULTS_PATH.exists():
    raise FileNotFoundError(f'Comparable MLP validation results not found: {MLP_RESULTS_PATH}')
if MODEL_OUTPUT_PATH.exists() or METRICS_OUTPUT_PATH.exists():
    raise FileExistsError('LSTM output path already exists; refusing to overwrite existing artifacts.')

print('TensorFlow:', tf.__version__)
print('Project directory:', PROJECT_DIR)
print('Random seed:', SEED)

TensorFlow: 2.21.0
Project directory: c:\Users\user\Desktop\DeepKey_DL_Project
Random seed: 42


In [3]:
# Recreate the corrected modeling data from the raw CSV; do not rely on prior notebook state.
df_fixed = pd.read_csv(DATA_PATH)
df_model = df_fixed.copy()

excluded_records = [
    ('p020', 2, 4),
    ('p071', 1, 20),
    ('p087', 1, 8),
]
for participant, session, repetition in excluded_records:
    record_mask = (
        df_model['participant'].eq(participant)
        & df_model['session'].eq(session)
        & df_model['repetition'].eq(repetition)
    )
    if int(record_mask.sum()) != 1:
        raise ValueError(f'Expected one excluded record: {(participant, session, repetition)}')
    df_model = df_model.loc[~record_mask].copy()

df_model.reset_index(drop=True, inplace=True)
excluded_columns = ['participant', 'session', 'repetition', 'total time']
feature_columns = [column for column in df_model.columns if column not in excluded_columns]
if len(df_model) != 19769 or len(feature_columns) != 46:
    raise ValueError(f'Unexpected corrected data shape/features: {df_model.shape}, {len(feature_columns)}')

session_1_mask = df_model['session'].eq(1)
X_session_1 = df_model.loc[session_1_mask, feature_columns].apply(pd.to_numeric, errors='raise')
y_session_1 = df_model.loc[session_1_mask, 'participant'].copy()
if len(X_session_1) != 9876 or y_session_1.nunique() != 99:
    raise ValueError(f'Unexpected Session 1 data: {X_session_1.shape}, participants={y_session_1.nunique()}')

print('Session 1 records:', len(X_session_1))
print('Features:', len(feature_columns))
print('Participants:', y_session_1.nunique())
print('Feature columns in original order:', feature_columns)

Session 1 records: 9876
Features: 46
Participants: 99
Feature columns in original order: ['DU.v.v', 'DD.v.p', 'DU.v.p', 'UD.v.p', 'UU.v.p', 'DU.p.p', 'DD.p.w', 'DU.p.w', 'UD.p.w', 'UU.p.w', 'DU.w.w', 'DD.w.j', 'DU.w.j', 'UD.w.j', 'UU.w.j', 'DU.j.j', 'DD.j.k', 'DU.j.k', 'UD.j.k', 'UU.j.k', 'DU.k.k', 'DD.k.e', 'DU.k.e', 'UD.k.e', 'UU.k.e', 'DU.e.e', 'DD.e.u', 'DU.e.u', 'UD.e.u', 'UU.e.u', 'DU.u.u', 'DD.u.r', 'DU.u.r', 'UD.u.r', 'UU.u.r', 'DU.r.r', 'DD.r.k', 'DU.r.k', 'UD.r.k', 'UU.r.k', 'DU.k.k.1', 'DD.k.b', 'DU.k.b', 'UD.k.b', 'UU.k.b', 'DU.b.b']


In [4]:
# Match the existing MLP validation split; Session 2 is not used below.
X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_session_1,
    y_session_1,
    test_size=0.20,
    random_state=SEED,
    stratify=y_session_1,
)

# Fit the class mapping on fit labels only, then apply it to validation labels.
label_encoder = LabelEncoder()
y_fit_encoded = label_encoder.fit_transform(y_fit)
unseen_validation_participants = set(y_validation.unique()) - set(label_encoder.classes_)
if unseen_validation_participants:
    raise ValueError(f'Validation classes absent from fit split: {sorted(unseen_validation_participants)}')
y_validation_encoded = label_encoder.transform(y_validation)
if len(label_encoder.classes_) != 99:
    raise ValueError(f'Expected 99 fit classes; found {len(label_encoder.classes_)}')

# Fit learned feature scaling exclusively on the fit partition.
scaler = StandardScaler()
X_fit_scaled = scaler.fit_transform(X_fit).astype(np.float32)
X_validation_scaled = scaler.transform(X_validation).astype(np.float32)
if scaler.n_samples_seen_ != len(X_fit):
    raise ValueError('Scaler was not fitted on the fit partition only.')

# One row is one timestep: there is no multi-keystroke temporal sequence in this table.
X_fit_sequence = X_fit_scaled.reshape(len(X_fit_scaled), 1, len(feature_columns))
X_validation_sequence = X_validation_scaled.reshape(
    len(X_validation_scaled), 1, len(feature_columns)
)
print('Fit/validation samples:', len(X_fit), len(X_validation))
print('Fit/validation sequence shapes:', X_fit_sequence.shape, X_validation_sequence.shape)
print('Encoded classes:', len(label_encoder.classes_), 'range:', int(y_fit_encoded.min()), 'to', int(y_fit_encoded.max()))
print('Scaler fit samples:', scaler.n_samples_seen_)


Fit/validation samples: 7900 1976
Fit/validation sequence shapes: (7900, 1, 46) (1976, 1, 46)
Encoded classes: 99 range: 0 to 98
Scaler fit samples: 7900.0


In [5]:
# Compact one-timestep LSTM multiclass classifier.
LSTM_UNITS = 64
DROPOUT_RATE = 0.20
LEARNING_RATE = 0.001
BATCH_SIZE = 256
MAX_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 10

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1, len(feature_columns))),
    tf.keras.layers.LSTM(LSTM_UNITS),
    tf.keras.layers.Dropout(DROPOUT_RATE),
    tf.keras.layers.Dense(len(label_encoder.classes_), activation='softmax'),
])
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='sparse_categorical_crossentropy',
    metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')],
)
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=EARLY_STOPPING_PATIENCE,
    restore_best_weights=True,
)
history = model.fit(
    X_fit_sequence,
    y_fit_encoded,
    validation_data=(X_validation_sequence, y_validation_encoded),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping],
    shuffle=True,
    verbose=2,
)
print('Epochs run:', len(history.history['loss']))
print('Best validation loss:', min(history.history['val_loss']))

Epoch 1/100
31/31 - 1s - 42ms/step - accuracy: 0.0424 - loss: 4.5365 - val_accuracy: 0.1088 - val_loss: 4.4678
Epoch 2/100
31/31 - 0s - 4ms/step - accuracy: 0.1252 - loss: 4.3917 - val_accuracy: 0.1650 - val_loss: 4.2998
Epoch 3/100
31/31 - 0s - 4ms/step - accuracy: 0.1691 - loss: 4.1945 - val_accuracy: 0.1938 - val_loss: 4.0825
Epoch 4/100
31/31 - 0s - 5ms/step - accuracy: 0.1842 - loss: 3.9675 - val_accuracy: 0.1999 - val_loss: 3.8528
Epoch 5/100
31/31 - 0s - 4ms/step - accuracy: 0.2023 - loss: 3.7385 - val_accuracy: 0.2308 - val_loss: 3.6394
Epoch 6/100
31/31 - 0s - 4ms/step - accuracy: 0.2282 - loss: 3.5331 - val_accuracy: 0.2540 - val_loss: 3.4512
Epoch 7/100
31/31 - 0s - 4ms/step - accuracy: 0.2622 - loss: 3.3488 - val_accuracy: 0.2768 - val_loss: 3.2849
Epoch 8/100
31/31 - 0s - 4ms/step - accuracy: 0.2930 - loss: 3.1932 - val_accuracy: 0.3041 - val_loss: 3.1369
Epoch 9/100
31/31 - 0s - 4ms/step - accuracy: 0.3151 - loss: 3.0458 - val_accuracy: 0.3305 - val_loss: 3.0042
Epoch 10/

In [6]:
def classification_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average='macro',
        zero_division=0,
    )
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_precision': precision,
        'macro_recall': recall,
        'macro_f1': f1,
    }

fit_probabilities = model.predict(X_fit_sequence, verbose=0)
validation_probabilities = model.predict(X_validation_sequence, verbose=0)
fit_predictions = fit_probabilities.argmax(axis=1)
validation_predictions = validation_probabilities.argmax(axis=1)
fit_metrics = classification_metrics(y_fit_encoded, fit_predictions)
validation_metrics = classification_metrics(y_validation_encoded, validation_predictions)

# Compare only existing Session 1 validation results; never load the saved Session 2 score.
mlp_results = pd.read_csv(MLP_RESULTS_PATH)
comparison_names = ['original_mlp_baseline', 'higher_alpha_0.01']
mlp_rows = mlp_results.loc[mlp_results['experiment'].isin(comparison_names)].copy()
if set(mlp_rows['experiment']) != set(comparison_names) or len(mlp_rows) != 2:
    raise ValueError('Expected exactly the two comparable MLP validation rows.')
if not (mlp_rows['split_random_state'].astype(int) == SEED).all():
    raise ValueError('Saved MLP validation rows do not use the matching split seed.')
if not (mlp_rows['validation_samples'].astype(int) == len(X_validation)).all():
    raise ValueError('Saved MLP validation rows do not match the current validation size.')

comparison_rows = []
for _, mlp_row in mlp_rows.iterrows():
    comparison_rows.append({
        'model': mlp_row['experiment'],
        'validation_accuracy': mlp_row['validation_accuracy'],
        'validation_balanced_accuracy': mlp_row['validation_balanced_accuracy'],
        'validation_macro_precision': mlp_row['validation_macro_precision'],
        'validation_macro_recall': mlp_row['validation_macro_recall'],
        'validation_macro_f1': mlp_row['validation_macro_f1'],
    })
comparison_rows.append({
    'model': 'lstm_one_timestep',
    **{f'validation_{name}': value for name, value in validation_metrics.items()},
})
comparison = pd.DataFrame(comparison_rows)

print('LSTM fit metrics:', {name: round(value, 4) for name, value in fit_metrics.items()})
print('LSTM validation metrics:', {name: round(value, 4) for name, value in validation_metrics.items()})
display(comparison)

# Save only new artifacts, and only after a successful fit and validation evaluation.
if MODEL_OUTPUT_PATH.exists() or METRICS_OUTPUT_PATH.exists():
    raise FileExistsError('LSTM output path appeared during execution; refusing to overwrite.')
model.save(MODEL_OUTPUT_PATH)
metrics_output = comparison.copy()
metrics_output['validation_split'] = 'stratified Session 1 80/20; random_state=42'
metrics_output['seed'] = SEED
metrics_output['sequence_shape'] = '(samples, 1, 46)'
metrics_output['lstm_units'] = LSTM_UNITS
metrics_output['dropout_rate'] = DROPOUT_RATE
metrics_output['learning_rate'] = LEARNING_RATE
metrics_output['batch_size'] = BATCH_SIZE
metrics_output['max_epochs'] = MAX_EPOCHS
metrics_output['early_stopping_patience'] = EARLY_STOPPING_PATIENCE
metrics_output.to_csv(METRICS_OUTPUT_PATH, index=False)
print('Saved model:', MODEL_OUTPUT_PATH)
print('Saved validation comparison:', METRICS_OUTPUT_PATH)

LSTM fit metrics: {'accuracy': 0.7847, 'balanced_accuracy': 0.7847, 'macro_precision': 0.7854, 'macro_recall': 0.7847, 'macro_f1': 0.7829}
LSTM validation metrics: {'accuracy': 0.6624, 'balanced_accuracy': 0.6623, 'macro_precision': 0.671, 'macro_recall': 0.6623, 'macro_f1': 0.6596}


,model,validation_accuracy,validation_balanced_accuracy,validation_macro_precision,validation_macro_recall,validation_macro_f1
0,original_mlp_baseline,0.667510,0.667225,0.670497,0.667225,0.663739
1,higher_alpha_0.01,0.678644,0.678416,0.683960,0.678416,0.675472
2,lstm_one_timestep,0.662449,0.662281,0.670978,0.662281,0.659591


Saved model: c:\Users\user\Desktop\DeepKey_DL_Project\models\deepkey_lstm_one_timestep.keras
Saved validation comparison: c:\Users\user\Desktop\DeepKey_DL_Project\results\metrics\lstm_one_timestep_validation.csv
